# Lab 03: Attention Seq2Seq

**Module 03** | Read `notes/03-attention-seq2seq.pdf` before running this notebook.

Implement additive attention in an encoder-decoder that copies reversed sequences, a classic test bed for alignment weights.

Run every cell top to bottom. Code is complete, no fill-in sections. Markdown cells explain what each block does and why.


## Runtime device

PyTorch can run on your NVIDIA GPU through CUDA or fall back to CPU. GPU execution moves matrix operations off the host and typically speeds up neural network training by a large factor. The next cell detects what is available and prints the active device so you know whether to expect fast training or should keep batch sizes small.


In [ ]:
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if device.type == "cuda":
    props = torch.cuda.get_device_properties(0)
    print(f"CUDA enabled, {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {props.total_memory / 1e9:.1f} GB")
else:
    print("CUDA not available, using CPU. Labs still run; training may take longer.")


Sequence-to-sequence models map an input sequence to an output sequence through an encoder and a decoder. Without attention, the encoder must compress all information into a single fixed-size vector, a bottleneck for longer inputs.


Additive (Bahdanau) attention removes that bottleneck by letting each decoder step form a weighted sum over all encoder hidden states. The weights are learned from the decoder state and each encoder position, producing an explicit alignment you can visualize.


### Step 1: The copy task and training pairs

We use a copy task: reverse each source string. Reversal is easy to verify and produces non-trivial alignments. Early decoder steps should attend to late encoder positions. The vocabulary is built from characters appearing in the small pair list below.


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import seaborn as sns
import torch
import torch.nn as nn
import torch.nn.functional as F

ROOT = Path("..").resolve()

PAD, SOS, EOS = 0, 1, 2
PAIRS = [
    ("hello", "olleh"),
    ("world", "dlrow"),
    ("ai", "ia"),
    ("seq", "qes"),
    ("copy", "ypoc"),
    ("data", "atad"),
]

chars = sorted({c for src, tgt in PAIRS for s in (src, tgt) for c in s})
char_to_idx = {ch: i + 3 for i, ch in enumerate(chars)}
idx_to_char = {i + 3: ch for i, ch in enumerate(chars)}
vocab_size = len(char_to_idx) + 3


def encode_word(word: str) -> list[int]:
    end_token = EOS
    return [SOS] + [char_to_idx[c] for c in word] + [end_token]


def decode_ids(ids: list[int]) -> str:
    out = []
    for idx in ids:
        if idx in (PAD, SOS):
            continue
        if idx == EOS:
            break
        out.append(idx_to_char[idx])
    return "".join(out)


src_seqs = [encode_word(s) for s, _ in PAIRS]
tgt_seqs = [encode_word(t) for _, t in PAIRS]
max_src = max(len(s) for s in src_seqs)
max_tgt = max(len(t) for t in tgt_seqs)

print(f"Training pairs: {len(PAIRS)} | Vocab: {vocab_size} | Max src/tgt len: {max_src}/{max_tgt}")
print("\nAll source-target pairs:")
print(f"{'#':<4} {'Source':<10} {'Target':<10} {'Task'}")
print("-" * 40)
for i, (src, tgt) in enumerate(PAIRS):
    print(f"{i:<4} {src:<10} {tgt:<10} reverse")


### Step 2: Encoder-decoder with Bahdanau attention

The encoder is a GRU that emits one hidden vector per source token. The decoder is another GRU, but before each step it computes Bahdanau attention: a feed-forward network scores every encoder position against the current decoder hidden state, softmax produces weights, and the context vector is their weighted sum over encoder outputs.

Teacher forcing feeds the ground-truth previous token during training, which speeds convergence on tiny datasets.


In [ ]:
EMBED_DIM = 64
HIDDEN_DIM = 128
EPOCHS = 20
LR = 1e-2


class Encoder(nn.Module):
    def __init__(self):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, EMBED_DIM, padding_idx=PAD)
        self.gru = nn.GRU(EMBED_DIM, HIDDEN_DIM, batch_first=True)

    def forward(self, src: torch.Tensor):
        embedded = self.embed(src)
        outputs, hidden = self.gru(embedded)
        return outputs, hidden


class BahdanauDecoder(nn.Module):
    def __init__(self):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, EMBED_DIM, padding_idx=PAD)
        self.gru = nn.GRU(EMBED_DIM + HIDDEN_DIM, HIDDEN_DIM, batch_first=True)
        self.W_enc = nn.Linear(HIDDEN_DIM, HIDDEN_DIM, bias=False)
        self.W_dec = nn.Linear(HIDDEN_DIM, HIDDEN_DIM)
        self.v = nn.Linear(HIDDEN_DIM, 1, bias=False)
        self.out = nn.Linear(HIDDEN_DIM, vocab_size)

    def _attention(self, decoder_hidden, encoder_outputs, src_mask):
        dec = self.W_dec(decoder_hidden).unsqueeze(1)
        enc = self.W_enc(encoder_outputs)
        energy = self.v(torch.tanh(dec + enc)).squeeze(-1)
        energy = energy.masked_fill(~src_mask, float("-inf"))
        weights = F.softmax(energy, dim=-1)
        context = torch.bmm(weights.unsqueeze(1), encoder_outputs).squeeze(1)
        return context, weights

    def forward(self, tgt_token, hidden, encoder_outputs, src_mask):
        embedded = self.embed(tgt_token)
        context, weights = self._attention(hidden.squeeze(0), encoder_outputs, src_mask)
        gru_in = torch.cat([embedded, context], dim=-1).unsqueeze(1)
        output, hidden = self.gru(gru_in, hidden)
        logits = self.out(output.squeeze(1))
        return logits, hidden, weights


class Seq2Seq(nn.Module):
    def __init__(self):
        super().__init__()
        self.encoder = Encoder()
        self.decoder = BahdanauDecoder()

    def forward(self, src, tgt, src_mask):
        enc_out, hidden = self.encoder(src)
        logits_list, attn_list = [], []
        dec_in = tgt[:, 0]
        for t in range(1, tgt.size(1)):
            logits, hidden, weights = self.decoder(dec_in, hidden, enc_out, src_mask)
            logits_list.append(logits)
            attn_list.append(weights)
            dec_in = tgt[:, t]
        return torch.stack(logits_list, dim=1), torch.stack(attn_list, dim=1)


def pad_batch(seqs, max_len):
    batch = torch.full((len(seqs), max_len), PAD, dtype=torch.long)
    mask = torch.zeros(len(seqs), max_len, dtype=torch.bool)
    for i, seq in enumerate(seqs):
        batch[i, : len(seq)] = torch.tensor(seq, dtype=torch.long)
        mask[i, : len(seq)] = True
    return batch, mask


src_batch, src_mask = pad_batch(src_seqs, max_src)
tgt_batch, _ = pad_batch(tgt_seqs, max_tgt)
src_batch = src_batch.to(device)
tgt_batch = tgt_batch.to(device)
src_mask = src_mask.to(device)

model = Seq2Seq().to(device)
criterion = nn.CrossEntropyLoss(ignore_index=PAD)
optimizer = torch.optim.Adam(model.parameters(), lr=LR)
n_params = sum(p.numel() for p in model.parameters())
print(model)
print(f"\nParameters: {n_params:,}")


### Step 3: Train the seq2seq model

Twenty epochs on six pairs is enough for the loss to approach zero when the model is sized correctly. Watch for steady decline. If loss stalls, attention weights often reveal misalignment rather than random guessing.


In [ ]:
loss_history = []

for epoch in range(1, EPOCHS + 1):
    model.train()
    optimizer.zero_grad()
    logits, _ = model(src_batch, tgt_batch, src_mask)
    loss = criterion(logits.reshape(-1, vocab_size), tgt_batch[:, 1:].reshape(-1))
    loss.backward()
    optimizer.step()
    loss_history.append(loss.item())
    if epoch == 1 or epoch % 5 == 0 or epoch == EPOCHS:
        print(f"Epoch {epoch:02d}/{EPOCHS}, loss {loss.item():.4f}")

plt.figure(figsize=(7, 4))
plt.plot(range(1, EPOCHS + 1), loss_history, marker="o")
plt.xlabel("Epoch")
plt.ylabel("Cross-entropy loss")
plt.title("Seq2Seq with Bahdanau attention")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


### Step 4: Evaluate all pairs

Greedy decoding runs the decoder step by step, always picking the highest-probability token. We compare predicted strings against targets for every pair in the training set.


In [ ]:
@torch.no_grad()
def decode_greedy(model, src_ids: list[int], max_len: int) -> str:
    model.eval()
    src = torch.tensor([src_ids], dtype=torch.long, device=device)
    mask = torch.ones(1, src.size(1), dtype=torch.bool, device=device)
    enc_out, hidden = model.encoder(src)
    token = torch.tensor([SOS], device=device)
    outputs = []
    for _ in range(max_len):
        logits, hidden, _ = model.decoder(token, hidden, enc_out, mask)
        next_id = logits.argmax(dim=-1)
        outputs.append(next_id.item())
        token = next_id
        if next_id.item() == EOS:
            break
    return decode_ids(outputs)


print("Evaluation on all pairs:")
print(f"{'Source':<10} {'Target':<10} {'Predicted':<10} {'Match'}")
print("-" * 44)
correct = 0
for src, tgt in PAIRS:
    src_ids = encode_word(src)
    pred = decode_greedy(model, src_ids, max_tgt)
    match = pred == tgt
    correct += int(match)
    mark = "OK" if match else "WRONG"
    print(f"{src:<10} {tgt:<10} {pred:<10} {mark}")

print(f"\nSequence accuracy: {correct}/{len(PAIRS)} ({100 * correct / len(PAIRS):.0f}%)")


### Step 5: Attention heatmap and alignment interpretation

For a reversal task the heatmap should show an anti-diagonal pattern: decoder step *i* attends to encoder position *n - i*. Dark cells along that diagonal mean the model learned to look at the correct source character when emitting each target character.

The example below uses the first pair (`hello` -> `olleh`).


In [ ]:
@torch.no_grad()
def decode_with_attention(model, src_ids, max_len):
    model.eval()
    src = torch.tensor([src_ids], dtype=torch.long, device=device)
    mask = torch.ones(1, src.size(1), dtype=torch.bool, device=device)
    enc_out, hidden = model.encoder(src)
    token = torch.tensor([SOS], device=device)
    outputs, weights = [], []
    for _ in range(max_len):
        logits, hidden, attn = model.decoder(token, hidden, enc_out, mask)
        next_id = logits.argmax(dim=-1)
        outputs.append(next_id.item())
        weights.append(attn.squeeze(0).cpu())
        token = next_id
        if next_id.item() == EOS:
            break
    return outputs, torch.stack(weights)


example_src = src_seqs[0]
pred_ids, attn = decode_with_attention(model, example_src, max_tgt)
src_word, tgt_word = PAIRS[0]
pred_word = decode_ids(pred_ids)

print(f"Source:    {src_word}")
print(f"Target:    {tgt_word}")
print(f"Predicted: {pred_word}")
print(f"\nAttention matrix shape: {tuple(attn.shape)} (decoder steps x encoder positions)")
print("\nAlignment interpretation:")
print("  For string reversal, expect high weights along the anti-diagonal.")
print("  Decoder step 0 should attend to the last source character,")
print("  decoder step 1 to the second-to-last, and so on.")
if attn.numel() > 0:
    peak_enc = attn.argmax(dim=-1).tolist()
    print(f"  Peak encoder index per decoder step: {peak_enc}")

fig, ax = plt.subplots(figsize=(8, 5))
sns.heatmap(attn.numpy(), cmap="Blues", ax=ax, cbar=True)
ax.set_xlabel("Encoder position")
ax.set_ylabel("Decoder step")
ax.set_title(f"Attention weights, '{src_word}' -> '{tgt_word}'")
plt.tight_layout()
plt.show()
